# Fast and Slow: Colab entry point

**Can a System One decision model (Kev-4B, an open JEV-style model) take over a planner's decidable calls, and how does it learn compared with classic RL?**
The spec, with its feature map and the reasoning behind each decision, is in `docs/spec.md`.

This notebook is deliberately thin: every cell calls one `fns` command from the tested `fast_and_slow` package.
**Colab will disconnect you; the study is built for it.** Colab Pro sessions end after roughly 24 hours at most, often sooner on an H100, and they drop after a stretch of idleness if the tab isn't kept open. Pro+ can keep running in the background. So:
- **Finished stages are skipped.** Every result is written to Google Drive.
- **Unfinished runs continue from their last checkpoint.**
  - Kev RL saves its adapter, optimizer and RNG state every 5 updates (about 5–10 minutes on an H100) and resumes exactly.
  - PPO saves every 25k decisions and at every learning-curve point.
- **After a disconnect:** reconnect, run sections 1–3 again, then the cell of the stage you were in. At most a few minutes of work are lost.

**Runtime:** GPU, H100 (Colab Pro). **Budget:** about 25 H100-hours in total (see the table in the spec).

| Stage | Where | Time (approx.) |
|---|---|---|
| Note bank (Phi-4-mini) | GPU | 10 min |
| References (Bayes / oracle / base-stock) | CPU | 2 min |
| Kev zero-shot + in-context | GPU | 30 min |
| PPO × 3 variants × 3 horizons × 2 seeds | CPU, **in the background**, while the GPU trains Kev | ~4 h |
| Kev + RL × 3 horizons × 2 seeds | GPU | ~18 h (split it across sessions) |
| Export the report | CPU | 1 min |

## 1. Settings

In [ ]:
REPO = "https://github.com/carlosdonado10/fast-and-slow.git"   # private repo: add a GITHUB_TOKEN secret (key icon, left sidebar)
BRANCH = "main"
PROFILE = "full"                                                # "smoke" runs everything in minutes with a fake Kev
RUNS = "/content/drive/MyDrive/fast-and-slow/runs"              # survives disconnects
HORIZONS = [1, 4, 12]
SEEDS = [0, 1]
PPO_VARIANTS = ["num", "emb", "sig"]

## 2. Setup: Drive, code, environment

In [ ]:
import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(RUNS, exist_ok=True)

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = REPO.replace("https://", f"https://{token}@") if token else REPO

if not os.path.exists("/content/fast-and-slow"):
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", url, "/content/fast-and-slow"], check=True)
else:
    subprocess.run(["git", "-C", "/content/fast-and-slow", "pull", "--ff-only"], check=True)
os.chdir("/content/fast-and-slow/backend")

In [ ]:
%%bash
cd /content/fast-and-slow/backend
command -v uv || curl -LsSf https://astral.sh/uv/install.sh | sh
export PATH="$HOME/.local/bin:$PATH"
uv sync --extra gpu --frozen 2>&1 | tail -3
nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

def fns(*args, background=False, log=None):
    """Run one study stage. Output streams here, or into a log file on Drive when background=True."""
    cmd = ["uv", "run", "--frozen", "fns", *map(str, args), "--profile", PROFILE, "--runs", RUNS]
    if background:
        f = open(log, "a")
        return subprocess.Popen(cmd, stdout=f, stderr=subprocess.STDOUT)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"fns {args[0]} failed")

fns("status")

## 3. Preflight
Loads Kev-4B on the GPU, scores one request and takes one gradient step. If this fails, don't start the long runs.

In [ ]:
fns("kev-check")

## 4. Notes and references

In [ ]:
fns("notes")   # Phi-4-mini writes the inbox notes once; the bank is frozen on Drive
fns("refs")    # random / base-stock / Bayes-optimal / perfect information on the held-out scenarios

## 5. Classic PPO, in the background on CPU
PPO doesn't need the GPU, so it runs alongside the Kev stages. Its log is written to Drive. Re-running this cell skips finished runs.

In [ ]:
script = " && ".join(
    f"uv run --frozen fns ppo --variant {v} --horizon {h} --seed {s} --profile {PROFILE} --runs {RUNS}"
    for s in SEEDS for h in HORIZONS for v in PPO_VARIANTS
)
ppo_log = f"{RUNS}/ppo.log"
ppo = subprocess.Popen(["bash", "-c", script], stdout=open(ppo_log, "a"), stderr=subprocess.STDOUT)
print("PPO running in the background, pid", ppo.pid, "- log:", ppo_log)

## 6. Kev zero-shot and in-context (no training)

In [ ]:
fns("kev")

## 7. Kev + RL
The long stage: hours per horizon and seed, so it will usually span several Colab sessions. Kev RL checkpoints to Drive every 5 updates
(`runs/<profile>/checkpoints/`). After a disconnect, re-run sections 1–3 and then this cell; it skips finished runs and resumes the interrupted one
from its last checkpoint. `fns("status")` shows progress at any time.

In [ ]:
for s in SEEDS:
    for h in HORIZONS:
        fns("kev-rl", "--horizon", h, "--seed", s)

In [ ]:
fns("status")   # what's done; safe to run any time


## 8. Wait for PPO, then export the report

In [ ]:
print("PPO still running" if ppo.poll() is None else f"PPO finished with code {ppo.returncode}")
ppo.wait()
print(open(ppo_log).read()[-2000:])
fns("status")

In [ ]:
report = f"{RUNS}/report.json"
fns("export", "--out", report)
from google.colab import files
files.download(report)   # put it at frontend/public/data/report.json, then `pnpm build`